# Data Preprocessing & Data Understanding

### Import the libraries

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

### Read The Survey Excel

In [2]:
file_path = r"Food_insecurity_Raw_Survey_Responses.xlsx"

xml = pd.ExcelFile(file_path)

print("Sheets found:", xml.sheet_names)

Sheets found: ['user_location', '2022-12-03_incentivised', '2022-11-22_27_incentivised', '2022-11-22_27_non_incentivised', '2022_12_05_all_responses_cleane', 'adult_household_fi', 'USDA_food_security_coding', 'Sheet1', 'Preliminary results']


### Load and merge survey sheets

In [3]:
df1 = pd.read_excel(file_path, sheet_name="2022-12-03_incentivised")
df2 = pd.read_excel(file_path, sheet_name="2022-11-22_27_incentivised")
df3 = pd.read_excel(file_path, sheet_name="2022-11-22_27_non_incentivised")

df = pd.concat([df1, df2, df3])

In [4]:
df.shape

(2891, 72)

## Remove non-consent records

In [5]:
df = df[
    df[
        "By agreeing to take part in this survey, you confirm that you have read, understood and agreed with the following statements. Please note that this survey will skip to the end if you disagree to take part: \n \n _I agree that data gathered in this study will be stored anonymously and securely, and will be used for research purposes only. _\n \n _I understand that my participation is voluntary and that I am free to withdraw at any time without giving reason. _\n \n _I understand that all personal information will remain confidentially within OLIOâ€™s database, and that no personally identifiable information will be shared with any third party, and that no data will be made available that can allow me to be personally identified in the results of this research. _\n \n _I am 18 years of age or older. _\n \n I agree to take part in this survey:"
    ]
    == 1
]

In [6]:
df.shape

(2717, 72)

In [7]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


#### Column standardization (rename long questions)

In [8]:
df.rename(
    columns={
        "The next questions are about your householdâ€™s diet in the past 12 months, since October of last year, and whether you were able to afford the food you needed. In the past 12 months, was the following statement true for you?\n \n We worried whether our food would run out before we got money to buy more.": "food_worry",
        "The food that we bought just didn't last, and we didn't have money to get more.": "food_not_last",
        "We couldnâ€™t afford to eat balanced meals.": "no_balanced_meal",
        "The next questions are about the food situation of your children. In the past 12 months, was the following statement true for you? \n \n We relied on only a few kinds of low-cost food to feed our children because we were running out of money to buy food.": "We relied on only a few kinds of low-cost food to feed our children because we were running out of money to buy food",
        "We couldnâ€™t feed our children a balanced meal, because we couldnâ€™t afford that.": "We couldn't feed our children a balanced meal, because we couldn't afford that.",
        "Did you ever cut the size of any of the children€™s meals because there wasn't enough money for food?": "Did you ever cut the size of any of the children's meals because there wasn't enough money for food?",
        "The following questions are about challenges your household may have had paying energy bills or maintaining heating in your home in the past 12 months. \n \n How frequently did your household reduce or forego expenses for basic household necessities, such as medicine or food, in order to pay an energy bill?": "How frequently did your household reduce or forego expenses for basic household necessities, such as medicine or food, in order to pay an energy bill?",
        "In the last year, was there ever a time your household was unable to use your main source of heat because you could not afford to pay for gas or electricity?": "In the last year, was there ever a time your household was unable to use your main source of heat because you could not afford to pay for gas or electricity?",
        "In the last year, was there ever a time your household was unable to use your main source of heat because the equipment was broken, and you couldnâ€™t afford to pay to repair or replace the equipment?": "In the last year, was there ever a time your household was unable to use your main source of heat because the equipment was broken, and you couldn't afford to pay to repair or replace the equipment?",
    },
    inplace=True,
)

#### Alias safeguard and domain question lists
#### Ensure this alias exists even if source text has apostrophe/encoding variations

In [9]:
if "food_not_last" not in df.columns:
    fallback_cols = [
        c
        for c in df.columns
        if "The food that we bought just" in c and "have money to get more" in c
    ]
    if fallback_cols:
        df.rename(columns={fallback_cols[0]: "food_not_last"}, inplace=True)

food_cols = [
    "food_worry",
    "food_not_last",
    "no_balanced_meal",
    "Did you or other adults in your household ever cut the size of your meals or skip meals because there wasn't enough money for food?",
    "How often did this happen?",
    "Did you ever eat less than you felt you should because there wasn't enough money for food?",
    "Were you ever hungry but didn't eat because there wasn't enough money for food?",
    "Did you lose weight because there wasn't enough money for food?",
    "Did you or other adults in your household ever not eat for a whole day because there wasn't enough money for food?",
    "How often did this happen?.1",
]

child_cols = [
    "We relied on only a few kinds of low-cost food to feed our children because we were running out of money to buy food",
    "We couldn't feed our children a balanced meal, because we couldn't afford that.",
    "The children were not eating enough because we just couldn't afford enough food.",
    "Did you ever cut the size of any of the children's meals because there wasn't enough money for food?",
    "Did any of the children ever skip meals because there wasn't enough money for food?",
    "Were the children ever hungry but you just couldn't afford more food?",
    "Did any of the children ever not eat for a whole day because there wasn't enough money for food?",
    "How often did this happen?.2",
]

fuel_cols = [
    "How frequently did your household reduce or forego expenses for basic household necessities, such as medicine or food, in order to pay an energy bill?",
    "In the past year, how frequently did your household keep your home at a cold temperature that you felt was unsafe or unhealthy?",
    "In the past year, how frequently did your household run behind on payments for energy bills, or receive a notice to disconnect?",
    "In the last year, was there ever a time your household was unable to use your main source of heat because you could not afford to pay for gas or electricity?",
    "About how many days over the past year has your household gone without heat because you could not afford to pay for gas or electricity?",
    "In the last year, was there ever a time your household was unable to use your main source of heat because the equipment was broken, and you couldn't afford to pay to repair or replace the equipment?",
    "In the past year, did anyone in your household need medical attention because your home was too cold?",
]

In [10]:
# Response mapping dictionary
mapping_dict = {
    "Often true": 1,
    "Sometimes true": 1,
    "Yes": 1,
    "Almost every month": 1,
    "Some months": 1,
    ">=36": 1,
    "Never true": 0,
    "No": 0,
    "Only 1 or 2 months": 0,
    "Never": 0,
    "<36": 0,
    "DK": 0,
}

In [11]:
# Combine domain columns
all_cols = food_cols + child_cols + fuel_cols

In [12]:
# Apply mapping and numeric conversion
for col in all_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace(mapping_dict)
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

In [13]:
# Validate mapped values (pre-threshold)
for col in all_cols:
    if col in df.columns:
        print(col, df[col].unique())

food_worry [0 1]
food_not_last [0.]
no_balanced_meal [0 1]
Did you or other adults in your household ever cut the size of your meals or skip meals because there wasn't enough money for food? [0. 1.]
How often did this happen? [0. 1.]
Did you ever eat less than you felt you should because there wasn't enough money for food? [0. 1.]
Were you ever hungry but didn't eat because there wasn't enough money for food? [0. 1.]
Did you lose weight because there wasn't enough money for food? [0. 1.]
Did you or other adults in your household ever not eat for a whole day because there wasn't enough money for food? [0. 1.]
How often did this happen?.1 [0.]
We relied on only a few kinds of low-cost food to feed our children because we were running out of money to buy food [0. 1.]
We couldn't feed our children a balanced meal, because we couldn't afford that. [0. 1.]
The children were not eating enough because we just couldn't afford enough food. [0. 1.]
Did you ever cut the size of any of the children

### Fuel heating-days threshold conversion

In [14]:
col_name = "About how many days over the past year has your household gone without heat because you could not afford to pay for gas or electricity?"

df[col_name] = df[col_name].fillna(0)

df[col_name] = df[col_name].apply(lambda x: 1 if float(x) >= 36 else 0)

In [15]:
# Validate mapped values (post-threshold)
for col in all_cols:
    if col in df.columns:
        print(col, df[col].unique())

food_worry [0 1]
food_not_last [0.]
no_balanced_meal [0 1]
Did you or other adults in your household ever cut the size of your meals or skip meals because there wasn't enough money for food? [0. 1.]
How often did this happen? [0. 1.]
Did you ever eat less than you felt you should because there wasn't enough money for food? [0. 1.]
Were you ever hungry but didn't eat because there wasn't enough money for food? [0. 1.]
Did you lose weight because there wasn't enough money for food? [0. 1.]
Did you or other adults in your household ever not eat for a whole day because there wasn't enough money for food? [0. 1.]
How often did this happen?.1 [0.]
We relied on only a few kinds of low-cost food to feed our children because we were running out of money to buy food [0. 1.]
We couldn't feed our children a balanced meal, because we couldn't afford that. [0. 1.]
The children were not eating enough because we just couldn't afford enough food. [0. 1.]
Did you ever cut the size of any of the children

In [16]:
# Build domain security scores
df["food_security_score"] = df[food_cols].sum(axis=1)
df["child_security_score"] = df[child_cols].sum(axis=1)
df["fuel_security_score"] = df[fuel_cols].sum(axis=1)

In [17]:
# Quick score sanity check
df["food_security_score"].unique()

array([0., 2., 5., 4., 3., 1., 7., 8., 6.])

In [18]:
# Incentivised distribution check
if "incentivised" in df.columns:
    print(df["incentivised"].value_counts(normalize=True))

incentivised
1    0.922341
0    0.077659
Name: proportion, dtype: float64


In [19]:
# Create adult food security label
def classify_adult_food(score):
    if score == 0:
        return "High"
    elif score <= 2:
        return "Marginal"
    elif score <= 5:
        return "Low"
    else:
        return "Very Low"


df["food_security_label"] = df["food_security_score"].apply(classify_adult_food)

In [20]:
# Create child food security label
def classify_child_food(score):
    if score == 0:
        return "High"
    elif score == 1:
        return "Marginal"
    elif score <= 4:
        return "Low"
    else:
        return "Very Low"


df["child_security_label"] = df["child_security_score"].apply(classify_child_food)

In [21]:
# Create fuel security label
def classify_fuel(score):
    if score == 0:
        return "Secure"
    elif score <= 2:
        return "Moderate Risk"
    else:
        return "High Risk"


df["fuel_security_label"] = df["fuel_security_score"].apply(classify_fuel)

In [22]:
# Normalize support-source indicators
df["Foodbank"] = df["Foodbank"].map({"Foodbank": 1}).fillna(0).astype(int)
df["Community or faith group"] = (
    df["Community or faith group"]
    .map({"Community or faith group": 1})
    .fillna(0)
    .astype(int)
)
df["Close family or friends"] = (
    df["Close family or friends"]
    .map({"Close family or friends": 1})
    .fillna(0)
    .astype(int)
)
df["Neighbours"] = df["Neighbours"].map({"Neighbours": 1}).fillna(0).astype(int)

In [23]:
# Export cleaned dataset
role_name = "role01"
output_file = f"cleaned_dataset_{role_name}.csv"
df.to_csv(output_file, index=False)
print(f"Saved {output_file}")

Saved cleaned_dataset_role01.csv
